# Embedding Layer

In [1]:
import torch
import requests

url = "https://raw.githubusercontent.com/lucasaraujomedeiros/llm-lab/main/frankenstein.txt"
file_path = "frankenstein.txt"

response = requests.get(url, timeout=30)
response.raise_for_status()

with open(file_path, "wb") as f:
    f.write(response.content)


In [2]:
with open("frankenstein.txt", encoding="utf-8") as f:
    raw_text = f.read()

In [3]:
def getVocab(text):
    tokens = sorted(set(tokenizingMethod(text)))
    return {string: num for (num,string) in enumerate(tokens)}


def tokenizingMethod(text):
    return text.split()

class Tokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.decodeDict = {string: num for (string, num) in vocab.items()}

    def encode(self, text):
        ids = [self.vocab[word] for word in tokenizingMethod(text)]
        return ids

    def decode(self, ids):
        words = [self.decodeDict[num] for num in ids]
        return words





Vamos criar uma camada de embedding

In [23]:
vocab = getVocab(raw_text)

vocab_size = len(vocab)
output_dim = 10

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)


#print(embedding_layer.weight[3].norm())
#print(embedding_layer.weight.sum(dim=1))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035,  ..., -0.3792,  0.7671, -1.1925],
        [ 0.6984, -1.4097,  0.1794,  ..., -1.0205, -0.1690,  0.9178],
        [ 1.5810,  1.3010,  1.2753,  ..., -1.1481, -1.1589,  0.3255],
        ...,
        [ 0.3973, -0.9922, -0.7078,  ..., -1.8333, -0.6585, -0.4464],
        [ 0.2626,  1.6548,  0.6471,  ...,  0.3141,  1.3080, -0.1319],
        [ 0.2449, -0.9826,  0.9710,  ..., -0.1128,  0.3763,  0.9788]],
       requires_grad=True)


Os valores da embedding_layer no momento são apenas números aleatórios, pois faz parte do treinamento setar os pesos da embedding layer.

Vale notar que uma das dimensões da embedding layer contém o número de palavras do vocabulário, justamente porque a operação de emmbedding é simplesmente traduzir cada token para sua linha correspondente na matriz dos pesos, obtendo um vetor com o número do output_dim estipulado.

In [5]:
#printar embedding dos tokens de text

text = raw_text[:50]
myTokenizer = Tokenizer(vocab)

ids = torch.tensor(myTokenizer.encode(text))
embedded_tokens = embedding_layer(ids)
print(embedded_tokens, "\n\n")


# printar cada linha da matriz de pesos correspondente ao ID dos tokens
#obs: note que são iguais

for id in ids:
  print(embedding_layer.weight[id])

tensor([[ 0.2031, -1.9496,  0.1636, -1.2704,  2.0420, -0.4635, -2.3387,  2.1219,
          0.3063,  0.4127],
        [-0.1499,  0.3924,  0.5455, -0.9749,  0.5502,  1.8038, -1.0277,  1.7130,
          0.1164, -1.0700],
        [ 0.1421,  0.5349,  1.5360,  1.5905, -0.2317,  0.5516,  0.2528,  0.7506,
         -0.2193, -1.3358],
        [-0.6034, -1.8413,  0.1591,  0.6083,  0.0133, -1.1690, -0.4273,  0.5029,
          1.4000,  0.1168],
        [ 0.9214, -1.0337,  0.3093,  2.4125,  1.4767,  1.0601,  2.0145,  2.3630,
         -0.0305, -0.0668],
        [ 1.3046, -1.2254, -1.2664,  0.6354, -0.3022,  1.0010, -1.2636,  1.3063,
         -0.8553, -1.5657],
        [-0.7982,  0.6608,  0.0173, -0.6489, -0.0206, -0.4319, -1.6442,  0.6075,
         -1.9661, -0.8284]], grad_fn=<EmbeddingBackward0>) 


tensor([ 0.2031, -1.9496,  0.1636, -1.2704,  2.0420, -0.4635, -2.3387,  2.1219,
         0.3063,  0.4127], grad_fn=<SelectBackward0>)
tensor([-0.1499,  0.3924,  0.5455, -0.9749,  0.5502,  1.8038, -1.0277

Para codificar também a posição de cada token na frase (ou no contexto), podemos utilizar uma segunda camada de embedding, onde escolheremos a linha da matriz de embedding usando o índice da palavra na frase, e somar o vetor obtido com o vetor do embedding do token.

Vale notar que, para a soma conseguir ser realizada, ambos os vetores tem de ter o mesmo número de dimensões (nesse caso, 10)



In [6]:
context_size = 4
positional_embedding_layer = torch.nn.Embedding(context_size, output_dim)
print(positional_embedding_layer.weight)

Parameter containing:
tensor([[-0.0342, -0.8970,  0.1022, -0.3130, -0.5100, -1.5038,  0.9896,  0.3345,
          0.0035, -2.6059],
        [ 1.5552,  0.4839,  1.8320, -1.2223, -0.1433,  1.7099,  1.2604,  0.3788,
         -0.7097, -0.1276],
        [-0.1128, -0.4024,  0.3584,  0.0227,  1.5917,  1.3183, -1.1748,  1.0716,
         -0.8663, -0.0502],
        [ 0.5837,  0.8683, -0.0739,  2.4441,  0.1927, -0.6429, -1.6309,  1.4258,
         -0.3424, -0.5279]], requires_grad=True)


# Attention

## Self Attention

No caso de self attention, nós usamos cada um dos tokens da frase como query, um de cada vez. Para cada query, nós fazemos um produto escalar (dot product) entre o vetor da query e os vetores dos inputs. O resultado operação é um número escalar que, de certa forma, representa a similaridade entre os dois vetores.

In [7]:
def dotProduct(tensor1, tensor2):
  scalar_out = 0
  for i in range(len(tensor1)):
    scalar_out += tensor1[i] * tensor2[i]

  return scalar_out

In [8]:
# Vamos calcular os attention_scores usando o token de indice 2 como query

query2 = embedded_tokens[2]

attention_scores = torch.zeros(len(embedded_tokens))

for i in range(len(embedded_tokens)):
  attention_scores[i] = dotProduct(embedded_tokens[i], query2)

print(attention_scores)

tensor([-3.1291,  2.7732,  8.0133, -0.7002,  6.5118,  2.1575,  0.5791],
       grad_fn=<CopySlices>)


Cada multiplicação dessa corresponde a um attention_score entre a query e um vetor de input. Podemos pegar a lista de attention_scores e normalizá-la, resultando em uma lista de attention_weights.

Quando fazemos esse processo, a soma de todos os attention_weights resulta em 1 e, quanto maior um attention_score, maior o attention_weight correspondente.

In [9]:
attention_weights = torch.softmax(attention_scores, dim=0)
print(attention_weights)

tensor([1.1760e-05, 4.3029e-03, 8.1186e-01, 1.3344e-04, 1.8089e-01, 2.3246e-03,
        4.7960e-04], grad_fn=<SoftmaxBackward0>)


Quando pegamos cada attention_weight, multiplicamos pelo seu token correspondente e, por fim, cada valor obtido, encontramos o valor do context vector

In [10]:
context_vector2 = torch.zeros(output_dim)
for i in range(len(embedded_tokens)):
  context_vector2 += attention_weights[i] * embedded_tokens[i]

print(context_vector2)

tensor([ 0.2839,  0.2461,  1.3024,  1.7247,  0.0807,  0.6493,  0.5614,  1.0476,
        -0.1858, -1.1052], grad_fn=<AddBackward0>)


## Self Attention Geral

In [11]:
attention_scores = embedded_tokens @ embedded_tokens.T
attention_weights = torch.softmax(attention_scores, dim=1)
context_vectors = attention_weights @ embedded_tokens

print(context_vectors, "\n\n")

#em uma linha:
context_vectors = (torch.softmax((embedded_tokens @ embedded_tokens.T), dim=1)) @ embedded_tokens

print(context_vectors)


tensor([[ 0.2031, -1.9496,  0.1636, -1.2704,  2.0420, -0.4635, -2.3386,  2.1219,
          0.3063,  0.4126],
        [-0.1351,  0.3278,  0.5278, -0.9718,  0.5819,  1.7405, -1.0600,  1.7192,
          0.1134, -1.0355],
        [ 0.2839,  0.2461,  1.3024,  1.7247,  0.0807,  0.6493,  0.5614,  1.0476,
         -0.1858, -1.1052],
        [-0.5108, -1.8505,  0.1595,  0.4066,  0.2385, -1.0855, -0.6315,  0.6847,
          1.2757,  0.1477],
        [ 0.9214, -1.0337,  0.3093,  2.4125,  1.4767,  1.0601,  2.0145,  2.3630,
         -0.0305, -0.0668],
        [ 1.3031, -1.2251, -1.2642,  0.6343, -0.2999,  1.0003, -1.2630,  1.3073,
         -0.8540, -1.5637],
        [-0.7878,  0.6462,  0.0172, -0.6493, -0.0116, -0.4213, -1.6436,  0.6183,
         -1.9480, -0.8262]], grad_fn=<MmBackward0>) 


tensor([[ 0.2031, -1.9496,  0.1636, -1.2704,  2.0420, -0.4635, -2.3386,  2.1219,
          0.3063,  0.4126],
        [-0.1351,  0.3278,  0.5278, -0.9718,  0.5819,  1.7405, -1.0600,  1.7192,
          0.1134, -1

## Self attention with trainable weights

Aqui, nós calculamos o attention score com base no dot product entre a query e a key de cada vetor. Calculamos os attention weights normalmente, normalizando os attention scores e, por fim, multiplicamos cada peso pelo value de cada token.

O ponto é que, tanto a query, a key e o value de cada token vão não são mais o valor do embedding de cada token e podem possuir, inclusive, um número diferente de dimensões que o embedding do token.

Para fazer a transformação do embedded_token -> query/key/value, que podem possuir outra dimensão dos embedded_tokens, fazemos uma transformação linear. Para isso, a matriz da transformação deve ser M x N, onde M representa o número de dimensões do token, e N representa o número de dimensões pretendido.

Podemos criar as matrizes de transformação para query, key e value com dimensão M x N e valores arbitrários, que serão treinados posteriormente.

In [12]:
token_dim = len(embedded_tokens[0])
output_dim = 5

#query = torch.nn.Embedding(token_dim, output_dim)
#print(query.weight.shape)

query_matrix = torch.nn.Parameter(torch.rand(token_dim, output_dim), requires_grad=False)
key_matrix = torch.nn.Parameter(torch.rand(token_dim, output_dim), requires_grad=False)
value_matrix = torch.nn.Parameter(torch.rand(token_dim, output_dim), requires_grad=False)

print(query_matrix.shape)


torch.Size([10, 5])


In [13]:
queries = embedded_tokens @ query_matrix
keys = embedded_tokens @ key_matrix
values = embedded_tokens @ value_matrix

Vamos achar os attention_scores. Eles vão ser calculados pelo dot product entre a query e a key. Depois, normalizamos os valores para achar os attention_weights e, por fim, multiplicamos cada attention_weight pelo seu respectivo value, para achar cada context vector.

In [14]:
attention_scores = queries @ keys.T
# fazendo a multiplicacao dessa forma, attention_scores[X][Y] vai ser o attention score da query do token[X]
# com a key do token[Y].

attention_weights = torch.softmax(attention_scores / output_dim ** 0.5, dim=1)
# dividimos pela raiz do número de dimensões das keys para minimizar o impacto de scores muito altos

print(attention_weights.shape, values.shape)
context_vectors = attention_weights @ values

torch.Size([7, 7]) torch.Size([7, 5])


Implementando isso numa classe, temos:

In [15]:
class SelfAttention():
  def __init__ (self, token_dim, output_dim):
    self.query_matrix = torch.nn.Parameter(torch.rand(token_dim, output_dim), requires_grad=False)
    self.key_matrix = torch.nn.Parameter(torch.rand(token_dim, output_dim), requires_grad=False)
    self.value_matrix = torch.nn.Parameter(torch.rand(token_dim, output_dim), requires_grad=False)

  def forward(self, tokens):
    queries = embedded_tokens @ self.query_matrix
    keys = embedded_tokens @ self.key_matrix
    values = embedded_tokens @ self.value_matrix

    attention_scores = queries @ keys.T
    attention_weights = torch.softmax(attention_scores / self.query_matrix.shape[1] ** 0.5, dim=1)
    context_vectors = attention_weights @ values

    return context_vectors


## Aplicando masking

Quando nossa LLM estiver tentando prever o próximo token, ela não pode ter informações dos tokens futuros da frase. Por isso, podemos aplicar um masking simples causal zerando os valores dos attention_weights que estão acima da diagonal principal.

In [16]:
context_length = attention_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])


Depois, fazemos a multiplicação termo a termo da nossa matriz mask_simple com a de attention_weights

In [17]:
masked_simple = attention_weights * mask_simple
print(masked_simple)

tensor([[2.7225e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.1785e-03, 3.9321e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.3640e-05, 7.2347e-04, 1.4250e-04, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [5.1470e-04, 8.3454e-05, 1.3828e-04, 9.1264e-03, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.1453e-14, 1.3879e-10, 2.1410e-12, 1.5020e-18, 1.0000e+00, 0.0000e+00,
         0.0000e+00],
        [3.0188e-06, 1.9193e-07, 2.9618e-07, 1.0660e-04, 1.9735e-12, 6.0525e-04,
         0.0000e+00],
        [1.2399e-07, 3.3341e-09, 2.8796e-08, 1.6652e-05, 2.9255e-14, 8.4061e-05,
         9.9990e-01]], grad_fn=<MulBackward0>)


Para garantir que os valores voltem a somar 1, podemos dividir cada attention_weight pela soma dos attention_weights daquela linha.

In [18]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.3060e-01, 7.6940e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.5507e-02, 8.2249e-01, 1.6200e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [5.2186e-02, 8.4615e-03, 1.4020e-02, 9.2533e-01, 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.1453e-14, 1.3879e-10, 2.1410e-12, 1.5020e-18, 1.0000e+00, 0.0000e+00,
         0.0000e+00],
        [4.2199e-03, 2.6830e-04, 4.1403e-04, 1.4902e-01, 2.7587e-09, 8.4608e-01,
         0.0000e+00],
        [1.2399e-07, 3.3341e-09, 2.8796e-08, 1.6652e-05, 2.9255e-14, 8.4061e-05,
         9.9990e-01]], grad_fn=<DivBackward0>)


Também podemos fazer o masking retirando attention_weights de maneira aleatória. Devemos multiplicar os attention_weights restantes por um fator de correção para compensar pelos que foram retirados.

In [19]:
dropout = torch.nn.Dropout(0.5)
print(dropout(attention_weights))

tensor([[5.4450e-02, 0.0000e+00, 1.0494e-01, 0.0000e+00, 1.7131e-02, 1.9639e-01,
         0.0000e+00],
        [0.0000e+00, 7.8642e-03, 7.3523e-03, 2.0410e-04, 1.9820e+00, 2.3262e-04,
         7.8617e-06],
        [0.0000e+00, 1.4469e-03, 0.0000e+00, 1.2006e-06, 1.9982e+00, 1.7176e-07,
         7.1809e-10],
        [1.0294e-03, 0.0000e+00, 2.7656e-04, 1.8253e-02, 0.0000e+00, 0.0000e+00,
         1.9566e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 3.0040e-18, 2.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 3.8386e-07, 5.9236e-07, 2.1321e-04, 0.0000e+00, 1.2105e-03,
         0.0000e+00],
        [2.4798e-07, 0.0000e+00, 0.0000e+00, 0.0000e+00, 5.8509e-14, 1.6812e-04,
         1.9998e+00]], grad_fn=<MulBackward0>)


## Multi-head attention

Para realizarmos o multi-head self attention, fazemos:

1. Se tivermos N "cabeças", teríamos N matrizes de W_key, W_query e W_value e basicamente faríamos o mesmo processo N vezes. No final, nosso context_vector seria a concatenação de cada context_vector gerado pela self attention de cada cabeça


In [21]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

